# Service Desk Problem clustering: Method comparison

Thin wrapper over the `problem_clustering` public API (same YAML as the CLI, no Method logic here).

1. Run Methods A/B/C (or load workbooks of earlier runs), with the Judge.
2. Compare their Metrics side by side.
3. Browse each Method's Problems with their Tickets.

Defaults: test fixture + Fake backend (offline). On Foundry: set `INPUT` to the export, `OVERRIDES = {}`
(or your `--set` equivalents) and the `FOUNDRY_*` env vars (see README).

In [ ]:
from pathlib import Path

import pandas as pd

from problem_clustering import compare, judge, load_config, load_tickets, run_method, write_outputs
from problem_clustering.backend import make_backend

ROOT = Path.cwd() if (Path.cwd() / 'pyproject.toml').exists() else Path.cwd().parent
INPUT = ROOT / 'tests' / 'fixtures' / 'tickets.xlsx'
CONFIG = ROOT / 'config.example.yaml'
# Dotted-key overrides, like CLI --set. Fake backend + no disk cache for the offline demo.
OVERRIDES = {'backend.kind': 'fake', 'runtime.cache_dir': None, 'method_a.distance_threshold': 0.6}
METHODS = ['a', 'b', 'c']
WORKBOOKS = {}  # method -> earlier run workbook to load instead of running, e.g. {'c': 'outputs/261115_1200_c.xlsx'}
RUN_JUDGE = True
OUT_DIR = ROOT / 'outputs'
BROWSE_PROBLEMS = None  # show only the N largest Problems per Method (None = all)
BROWSE_TICKETS = 20  # Tickets listed per Problem

## 1. Run or load each Method

In [ ]:
config = load_config(CONFIG, OVERRIDES)
workbooks = {m: Path(WORKBOOKS[m]) for m in METHODS if m in WORKBOOKS}
to_run = [m for m in METHODS if m not in workbooks]
if to_run:
    tickets = load_tickets(INPUT)
    backend = make_backend(config.backend)
for method in to_run:
    result = run_method(tickets, method, config, backend)
    judge_result = judge(result, config, backend) if RUN_JUDGE else None
    workbooks[method] = write_outputs(result, judge_result, OUT_DIR)['xlsx']
    print(result.log[-1])
workbooks = {m: workbooks[m] for m in METHODS}
for method, path in workbooks.items():
    print(method, path)

## 2. Comparison (Metrics sheets side by side)

In [ ]:
comparison_path = compare(list(workbooks.values()), OUT_DIR)
comparison = pd.read_excel(comparison_path, sheet_name='Comparison', index_col='metric')
print(comparison_path)
display(comparison)

Judge verdicts per Method (audit the Mixed / Split checks):

In [ ]:
for method, path in workbooks.items():
    checks = pd.read_excel(path, sheet_name='Judge')
    print(f'--- Method {method.upper()}: {len(checks)} Judge checks')
    display(checks)

## 3. Problem browser

In [ ]:
def browse(method, problems=BROWSE_PROBLEMS, tickets_per_problem=BROWSE_TICKETS):
    path = workbooks[method]
    catalogue = pd.read_excel(path, sheet_name='Problem Catalogue')
    tickets_sheet = pd.read_excel(path, sheet_name='Tickets')
    print(f'===== Method {method.upper()}: {len(catalogue)} Problems ({Path(path).name}) =====')
    shown = catalogue if problems is None else catalogue.head(problems)
    for p in shown.itertuples():
        print(f'\n[{p.problem_id}] {p.problem_name} ({p.ticket_count} Tickets)')
        print(f'    {p.description}')
        members = tickets_sheet[tickets_sheet['problem_id'] == p.problem_id]
        for _, t in members.head(tickets_per_problem).iterrows():
            title = '' if pd.isna(t['Tytuł']) else t['Tytuł']
            print(f"    - {t['Numer I/SR']}  {title}  (confidence {t['confidence']:.2f})")


for method in workbooks:
    browse(method)

Drill into one Problem's Tickets with all original columns:

In [ ]:
method, problem_id = METHODS[0], None  # e.g. 'a', 'A-0001'; None = largest Problem
tickets_sheet = pd.read_excel(workbooks[method], sheet_name='Tickets')
problem_id = problem_id or tickets_sheet['problem_id'].value_counts().index[0]
display(tickets_sheet[tickets_sheet['problem_id'] == problem_id])